# Notebook 2 – Regularization

**Dataset:** Online Retail transactions (`data.csv`)

We use two small tasks:
- **Classification** (is the order from the UK?) to demonstrate L1 / L2 / Elastic Net in Logistic Regression.
- **Regression** (predict `UnitPrice` from `Quantity` + some noise features) to demonstrate Ridge and Lasso Regression.

## Setup: Load Data
Load the CSV and do light cleaning, used by both tasks below.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(5000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.shape

(5000, 9)

## 1. What is Regularization?
**Regularization** adds a penalty to the model based on the size of its coefficients. This discourages the model from relying too heavily on any one feature, keeping it simpler.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
plain_model = LogisticRegression(penalty=None, max_iter=1000).fit(X_train, y_train)
print("No regularization — Train:", accuracy_score(y_train, plain_model.predict(X_train)),
      "Test:", accuracy_score(y_test, plain_model.predict(X_test)))

No regularization — Train: 0.89025 Test: 0.891


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 2. Why Regularization?
Without it, models (especially with many/correlated features) can **overfit** — fitting noise in the training data. Regularization reduces this by shrinking coefficients, trading a little training accuracy for better generalization.

In [4]:
reg_model = LogisticRegression(penalty='l2', C=0.1, max_iter=1000).fit(X_train, y_train)
print("No-reg coefficients:", plain_model.coef_)
print("Regularized coefficients:", reg_model.coef_)

No-reg coefficients: [[-0.00151041 -0.02596833 -0.00063129]]
Regularized coefficients: [[-0.00150951 -0.02592163 -0.0006317 ]]


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 3. L1 Regularization
**L1** (Lasso-style) penalizes the **absolute value** of coefficients. It can shrink some coefficients all the way to **zero**, effectively removing those features.

In [5]:
l1_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1).fit(X_train, y_train)
print("L1 coefficients:", l1_model.coef_)
print("Test accuracy:", accuracy_score(y_test, l1_model.predict(X_test)))

L1 coefficients: [[-0.00129424 -0.0201066  -0.0006677 ]]
Test accuracy: 0.891


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


## 4. L2 Regularization
**L2** (Ridge-style) penalizes the **squared value** of coefficients. It shrinks coefficients toward zero but rarely makes them exactly zero — all features stay, just smaller.

In [6]:
l2_model = LogisticRegression(penalty='l2', C=0.1, max_iter=1000).fit(X_train, y_train)
print("L2 coefficients:", l2_model.coef_)
print("Test accuracy:", accuracy_score(y_test, l2_model.predict(X_test)))

L2 coefficients: [[-0.00150951 -0.02592163 -0.0006317 ]]
Test accuracy: 0.891


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 5. Elastic Net
**Elastic Net** combines L1 and L2 penalties together (controlled by `l1_ratio`). It gets some feature-elimination from L1 and some stability from L2.

In [7]:
elastic_model = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.1, max_iter=2000).fit(X_train, y_train)
print("Elastic Net coefficients:", elastic_model.coef_)
print("Test accuracy:", accuracy_score(y_test, elastic_model.predict(X_test)))

C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Elastic Net coefficients: [[ 0.04009675  0.25195862 -0.00819325]]
Test accuracy: 0.891


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## 6. Ridge Regression
**Ridge Regression** is L2 regularization applied to a regression problem. Here we predict `UnitPrice` from `Quantity` plus a few random noise features, to see how Ridge shrinks all coefficients (without zeroing them).

In [8]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import train_test_split as tts

np.random.seed(42)
reg_df = df.sample(2000, random_state=1).copy()
reg_df['Noise1'] = np.random.randn(len(reg_df))
reg_df['Noise2'] = np.random.randn(len(reg_df))

Xr = reg_df[['Quantity', 'Noise1', 'Noise2']]
yr = reg_df['UnitPrice']
Xr_train, Xr_test, yr_train, yr_test = tts(Xr, yr, test_size=0.2, random_state=42)

ridge = Ridge(alpha=1.0).fit(Xr_train, yr_train)
print("Ridge coefficients:", ridge.coef_)

Ridge coefficients: [-0.00998249 -0.26790345  0.38703835]


## 7. Lasso Regression
**Lasso Regression** is L1 regularization applied to a regression problem. Watch how the noise feature coefficients shrink to (near) **zero** — Lasso automatically ignores useless features.

In [9]:
lasso = Lasso(alpha=0.1).fit(Xr_train, yr_train)
print("Lasso coefficients:", lasso.coef_)
print("(Quantity, Noise1, Noise2)")

Lasso coefficients: [-0.01000015 -0.16548951  0.28957634]
(Quantity, Noise1, Noise2)


## 8. Regularization Parameter
The strength of regularization is controlled by a parameter: `C` in Logistic Regression (smaller `C` = stronger penalty) or `alpha` in Ridge/Lasso (larger `alpha` = stronger penalty). Choosing it well is key — too strong causes underfitting, too weak doesn't help with overfitting.

In [10]:
for c in [0.01, 1, 100]:
    m = LogisticRegression(penalty='l2', C=c, max_iter=1000).fit(X_train, y_train)
    print(f"C={c}: Train={accuracy_score(y_train, m.predict(X_train)):.3f}, Test={accuracy_score(y_test, m.predict(X_test)):.3f}")

C=0.01: Train=0.890, Test=0.891
C=1: Train=0.890, Test=0.891
C=100: Train=0.890, Test=0.891


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 9. Feature Selection through L1
Because L1 can zero out coefficients entirely, it doubles as an automatic **feature selection** method — features with a zero coefficient can simply be dropped.

In [11]:
strong_lasso = Lasso(alpha=1.0).fit(Xr_train, yr_train)
selected = [name for name, coef in zip(Xr.columns, strong_lasso.coef_) if coef != 0]
print("Coefficients:", strong_lasso.coef_)
print("Features selected (non-zero):", selected)

Coefficients: [-0.00974296 -0.          0.        ]
Features selected (non-zero): ['Quantity']


## 10. Overfitting Reduction
Comparing a regularized vs. non-regularized model's train-test gap shows regularization doing its job: a smaller gap means less overfitting.

In [12]:
for name, m in [('No Regularization', plain_model), ('L2 (C=0.1)', reg_model)]:
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"{name}: Train={tr:.3f}, Test={te:.3f}, Gap={tr-te:.3f}")

No Regularization: Train=0.890, Test=0.891, Gap=-0.001
L2 (C=0.1): Train=0.890, Test=0.891, Gap=-0.001
